# LightGBM Volatility Backtest

CPU-only tree-based walk-forward backtest. LightGBM handles NaN natively,
so no scaling or missing-value imputation is needed for features.

In [ ]:
# ── Setup: clone repo, install deps ──
import os

REPO_URL = "https://github.com/jamesdchen/harxhar.git"
REPO_DIR = "/content/harxhar"
if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
!pip install -q numpy pandas scikit-learn pyarrow tqdm lightgbm

import sys
sys.path.insert(0, "/content/harxhar/colab")

In [ ]:
# ── Parameters ──
HORIZON = 1
TRAIN_WINDOW_DAYS = 500
PERIODS_PER_DAY = 48
REFIT_FREQUENCY = 5
DATA_PATH = "all30min"

In [ ]:
# ── Load + transform ──
import numpy as np
import pandas as pd

from src.loading import load_raw_data
from src.transforms import robust_transform

df = load_raw_data(DATA_PATH, allow_missing=True)
print(f"Loaded: {df.shape}")

# Full transform on RV target (diurnal + winsor)
adj_rv, baseline = robust_transform(
    df, "RV", is_target=True, use_diurnal=True, winsor_window=240
)
df["adj_RV"] = adj_rv
df["baseline"] = baseline

print(f"adj_RV stats:\n{df['adj_RV'].describe()}")

In [ ]:
# ── HAR features + DOW + hour ──

def resolve_har_lags(max_lag=3125):
    seq, v = [], 1
    while v <= max_lag:
        seq.append(v)
        v *= 5
    return seq

lags = resolve_har_lags()
har_names = []
for lag in lags:
    name = f"har_ma_{lag}"
    df[name] = df["adj_RV"].rolling(window=lag, min_periods=1).mean().shift(1)
    har_names.append(name)

# Calendar features
df["DOW"] = df["t"].dt.dayofweek
df["hour"] = df["t"].dt.hour
cal_names = ["DOW", "hour"]

feature_names = har_names + cal_names
print(f"Features ({len(feature_names)}): {feature_names}")

# Drop initial NaN rows from HAR computation
max_lag = lags[-1]
df = df.iloc[max_lag:].reset_index(drop=True)
print(f"Shape after lag trim: {df.shape}")

In [ ]:
# ── LightGBM model + walk-forward backtest ──
from lightgbm import LGBMRegressor
from tqdm import tqdm

# Extract arrays
X = df[feature_names].values.astype(np.float64)
y = df["adj_RV"].values.astype(np.float64)
dates = df["t"]
baselines = df["baseline"].values.astype(np.float64)

# Horizon shift
if HORIZON > 1:
    shift = HORIZON - 1
    X = X[:-shift]
    y = y[shift:]
    dates = dates.iloc[:-shift].reset_index(drop=True)
    baselines = baselines[shift:]

train_win = TRAIN_WINDOW_DAYS * PERIODS_PER_DAY
n_samples, n_features = X.shape
print(f"Samples: {n_samples}, Features: {n_features}, Train window: {train_win}")

# RollingBuffer
class RollingBuffer:
    def __init__(self, window_size, n_features):
        self.window_size = window_size
        self.ptr = 0
        self.count = 0
        self.X_buffer = np.zeros((window_size, n_features))
        self.y_buffer = np.zeros((window_size, 1))

    def add(self, x_new, y_new):
        self.X_buffer[self.ptr] = x_new
        self.y_buffer[self.ptr] = y_new
        self.ptr = (self.ptr + 1) % self.window_size
        if self.count < self.window_size:
            self.count += 1

    def get_view(self):
        return self.X_buffer, self.y_buffer

# Walk-forward
buf = RollingBuffer(train_win, n_features)
for i in range(train_win):
    buf.add(X[i], y[i:i+1])

model_fn = lambda: LGBMRegressor(
    n_estimators=500, max_depth=5, learning_rate=0.1,
    n_jobs=-1, verbose=-1,
)

X_buf, y_buf = buf.get_view()
model = model_fn()
model.fit(X_buf, y_buf.ravel())

predictions = np.full(n_samples - train_win, np.nan)
for t in tqdm(range(train_win, n_samples), desc="LightGBM backtest"):
    x_t = X[t]
    predictions[t - train_win] = model.predict(x_t.reshape(1, -1))[0]
    buf.add(x_t, y[t:t+1])
    if (t - train_win + 1) % REFIT_FREQUENCY == 0:
        X_buf, y_buf = buf.get_view()
        model = model_fn()
        model.fit(X_buf, y_buf.ravel())

print(f"Predictions: {predictions.shape}, NaN count: {np.isnan(predictions).sum()}")

In [ ]:
# ── Feature importance ──
import matplotlib.pyplot as plt

importance = model.feature_importances_
sorted_idx = np.argsort(importance)

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh([feature_names[i] for i in sorted_idx], importance[sorted_idx])
ax.set_xlabel("Feature Importance (split count)")
ax.set_title("LightGBM Feature Importance")
plt.tight_layout()
plt.show()

In [ ]:
# ── Quick eval ──
oos_start = train_win
y_oos = y[oos_start:]
dates_oos = dates.iloc[oos_start:].values
baselines_oos = baselines[oos_start:]

# Duan smearing
smear = np.mean((y_oos - predictions) ** 2)
pred_raw = (predictions**2 + smear) * baselines_oos
true_raw = (y_oos**2) * baselines_oos

# Adjusted-scale metrics
errors = y_oos - predictions
mse = np.mean(errors**2)
mae = np.mean(np.abs(errors))

# QLIKE (raw scale)
mask = (true_raw > 0) & (pred_raw > 0)
ratio = true_raw[mask] / pred_raw[mask]
qlike = np.mean(ratio - np.log(ratio) - 1.0)

print(f"MSE (adj):  {mse:.6f}")
print(f"MAE (adj):  {mae:.6f}")
print(f"QLIKE:      {qlike:.6f}")
print(f"OOS samples: {len(y_oos):,}")

In [ ]:
%%writefile ../src/ml_lightgbm.py
"""LightGBM volatility backtest executor.

Self-contained walk-forward backtest with HAR lag features, DOW/hour
features, and Duan smearing.  No imports from core/ or projects/.
Tree-based model: no scaling, handles NaN natively.
"""

import argparse
import os

import numpy as np
import pandas as pd
from lightgbm import LGBMRegressor
from tqdm import tqdm

from src.loading import load_raw_data
from src.transforms import robust_transform

# ── Constants ─────────────────────────────────────────────────────────────
PERIODS_PER_DAY = 48


# ── HAR lag features ─────────────────────────────────────────────────────

def resolve_har_lags(max_lag: int = 3125) -> list[int]:
    seq, v = [], 1
    while v <= max_lag:
        seq.append(v)
        v *= 5
    return seq


def generate_har_features(
    df: pd.DataFrame, target_col: str = "adj_RV"
) -> tuple[pd.DataFrame, list[str]]:
    lags = resolve_har_lags()
    features: dict[str, pd.Series] = {}
    feature_names: list[str] = []
    for lag in lags:
        name = f"har_ma_{lag}"
        features[name] = (
            df[target_col].rolling(window=lag, min_periods=1).mean().shift(1)
        )
        feature_names.append(name)
    feat_df = pd.DataFrame(features, index=df.index)
    return pd.concat([df, feat_df], axis=1), feature_names


# ── DOW + hour features ─────────────────────────────────────────────────

def add_calendar_features(df: pd.DataFrame) -> list[str]:
    """Add day-of-week (0-6) and hour features. Returns new column names."""
    df["DOW"] = df["t"].dt.dayofweek
    df["hour"] = df["t"].dt.hour
    return ["DOW", "hour"]


# ── Horizon shift ─────────────────────────────────────────────────────────

def apply_horizon_shift(
    X: np.ndarray,
    y: np.ndarray,
    dates: pd.Series,
    baselines: np.ndarray,
    horizon: int,
) -> tuple[np.ndarray, np.ndarray, pd.Series, np.ndarray]:
    if horizon <= 1:
        return X, y, dates, baselines
    shift = horizon - 1
    return (
        X[:-shift],
        y[shift:],
        dates.iloc[:-shift].reset_index(drop=True),
        baselines[shift:],
    )


# ── RollingBuffer ─────────────────────────────────────────────────────────

class RollingBuffer:
    """Ring buffer for (X, y) pairs."""

    def __init__(self, window_size: int, n_features: int) -> None:
        self.window_size = window_size
        self.ptr = 0
        self.count = 0
        self.X_buffer = np.zeros((window_size, n_features))
        self.y_buffer = np.zeros((window_size, 1))

    def add(self, x_new: np.ndarray, y_new: np.ndarray) -> None:
        self.X_buffer[self.ptr] = x_new
        self.y_buffer[self.ptr] = y_new
        self.ptr = (self.ptr + 1) % self.window_size
        if self.count < self.window_size:
            self.count += 1

    def get_view(self) -> tuple[np.ndarray, np.ndarray]:
        return self.X_buffer, self.y_buffer


# ── Duan smearing (inline) ───────────────────────────────────────────────

def apply_duan_smearing(
    forecasts: np.ndarray, y_true: np.ndarray, baselines: np.ndarray
) -> tuple[np.ndarray, np.ndarray]:
    smear = np.mean((y_true - forecasts) ** 2)
    pred_raw = (forecasts**2 + smear) * baselines
    true_raw = (y_true**2) * baselines
    return pred_raw, true_raw


# ── Walk-forward backtest ─────────────────────────────────────────────────

def run_backtest(
    model_fn,
    X: np.ndarray,
    y: np.ndarray,
    train_win: int,
    refit_frequency: int = 5,
) -> np.ndarray:
    """Walk-forward backtest returning an array of predictions."""
    n_samples, n_features = X.shape
    predictions = np.full(n_samples - train_win, np.nan)

    buf = RollingBuffer(train_win, n_features)
    for i in range(train_win):
        buf.add(X[i], y[i : i + 1])

    X_buf, y_buf = buf.get_view()
    model = model_fn()
    model.fit(X_buf, y_buf.ravel())

    for t in tqdm(range(train_win, n_samples), desc="backtest"):
        x_t = X[t]
        predictions[t - train_win] = model.predict(x_t.reshape(1, -1))[0]
        buf.add(x_t, y[t : t + 1])
        if (t - train_win + 1) % refit_frequency == 0:
            X_buf, y_buf = buf.get_view()
            model = model_fn()
            model.fit(X_buf, y_buf.ravel())

    return predictions


# ── CLI ───────────────────────────────────────────────────────────────────

def main() -> None:
    parser = argparse.ArgumentParser(description="LightGBM walk-forward backtest")
    parser.add_argument("--data-path", default="all30min")
    parser.add_argument("--horizon", type=int, default=1)
    parser.add_argument(
        "--train-window", type=int, default=500, help="training window in days"
    )
    parser.add_argument("--start", type=int, default=0)
    parser.add_argument("--end", type=int, default=-1)
    parser.add_argument("--output-file", required=True)
    args = parser.parse_args()

    train_win_periods = args.train_window * PERIODS_PER_DAY

    df = load_raw_data(args.data_path, allow_missing=True)

    adj_rv, baseline = robust_transform(
        df, "RV", is_target=True, use_diurnal=True, winsor_window=240
    )
    df["adj_RV"] = adj_rv
    df["baseline"] = baseline

    df, har_names = generate_har_features(df, target_col="adj_RV")
    cal_names = add_calendar_features(df)
    feature_names = har_names + cal_names

    max_lag = resolve_har_lags()[-1]
    df = df.iloc[max_lag:].reset_index(drop=True)

    X = df[feature_names].values.astype(np.float64)
    y = df["adj_RV"].values.astype(np.float64)
    dates = df["t"]
    baselines = df["baseline"].values.astype(np.float64)

    X, y, dates, baselines = apply_horizon_shift(X, y, dates, baselines, args.horizon)

    start = args.start
    end = len(X) if args.end == -1 else args.end

    X_chunk = X[start:end]
    y_chunk = y[start:end]
    dates_chunk = dates.iloc[start:end].reset_index(drop=True)
    baselines_chunk = baselines[start:end]

    if train_win_periods >= len(X_chunk):
        raise ValueError(
            f"train_window ({train_win_periods} periods) >= chunk size ({len(X_chunk)})"
        )

    model_fn = lambda: LGBMRegressor(
        n_estimators=500, max_depth=5, learning_rate=0.1,
        n_jobs=-1, verbose=-1,
    )
    preds = run_backtest(
        model_fn, X_chunk, y_chunk,
        train_win=train_win_periods, refit_frequency=5,
    )

    oos_start = train_win_periods
    y_oos = y_chunk[oos_start:]
    dates_oos = dates_chunk.iloc[oos_start:].values
    baselines_oos = baselines_chunk[oos_start:]

    pred_raw, true_raw = apply_duan_smearing(preds, y_oos, baselines_oos)

    results = pd.DataFrame({
        "date": dates_oos, "horizon": args.horizon,
        "true_adj": y_oos, "pred_adj": preds,
        "true_raw": true_raw, "pred_raw": pred_raw,
    })

    os.makedirs(os.path.dirname(args.output_file) or ".", exist_ok=True)
    results.to_csv(args.output_file, index=False)
    print(f"Saved {len(results)} rows -> {args.output_file}")


if __name__ == "__main__":
    main()